<a href="https://colab.research.google.com/github/anisha0325/MedQA/blob/main/MedQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
filepath = "/content/drive/MyDrive/newsample.txt"
with open(file_path, 'r') as file:
    content = file.readlines()

In [ ]:
contexts = []
for elem in content:
  if 'Context' in elem:
    contexts.append(elem.strip())
print(len(contexts))

500


In [ ]:
def split_on_context(lst):
    result = []
    current_context = None

    for item in lst:
        if item.startswith('[Context '):
            # If we find a new context, start a new sublist
            if current_context is not None:
                result.append(current_context)
            current_context = [item.strip()]
        else:
            # Otherwise, append the item to the current context list
            if current_context is not None and item != '\n':
                current_context.append(item.strip())

    # Don't forget to append the last context list to the result
    if current_context is not None:
        result.append(current_context)

    return result

# Example usage:
result = split_on_context(content)
print(len(result))


500


In [ ]:
def list_to_dict(lst):
    current_id = None
    current_entry = {}
    result = {}
    for item in lst:
        key, value = item.split(' --> ')
        key = key.strip()
        value = value.strip()

        if key == 'QID':
            # When encountering a new ID, store the current entry if it exists
            if current_id is not None:
                result[current_id] = current_entry
            # Start a new entry
            current_id = value
            current_entry = {}
        else:
            # For 'que' and 'ans', add to the current entry
            current_entry[key] = value

    # Don't forget to add the last entry to the result
    if current_id is not None:
        result[current_id] = current_entry

    return result

context_qa_list = []
for res in result:
  context_dict = {res[0]: res[1]}
  qa_dict =  list_to_dict(res[2:])
  context_qa_list.append({**context_dict, **qa_dict})

In [ ]:
import pickle

file_path = '/content/drive/MyDrive/MedQA/context_qa_list.pkl'

# Dump the list to a pickle file
with open(file_path, 'wb') as file:
    pickle.dump(context_qa_list, file)

In [ ]:
context_qa_list[0]

{'[Context 1]': "var s_context; s_context= s_context || {}; s_context['wb.modimp'] = 'vidfloat'; if(webmd.useragent && webmd.useragent.ua.type === 'desktop'){ webmd.ads2.disable Initial Load(); webmd.ads2.disable Ads Init = true; $(function() { webmd.p.pim.increment(); $('.responsive-video-container').insert After('.module-social-share-container'); require(['video2/1/responsive-player/video-loader'], function(video Loader) { video Loader.init({ autoplay: webmd.useragent.ua.type === 'desktop' && ! !s_sensitive, chron ID: $('article embeded_module[type=video][align=top]:eq(0)').attr('chronic_id'), continuous Play: true, cp Options: { flyout: true }, display Ads: true, mode: 'in-article', sticky: true }) }); }); } else { $(function(){ $('.responsive-video-container').remove(); }); } Gastritis is an inflammation, irritation, or erosion of the lining of the stomach. It can occur suddenly (acute) or gradually (chronic). Gastritis can be caused by irritation due to excessive alcohol use, chro

In [ ]:
from tqdm import tqdm
QID_context_q = dict()
QID_ans = dict()
for elem in tqdm(context_qa_list):
  for key, value in elem.items():
    if 'Context' in key:
      present_context = value
    else:
      QID = key
      ques = value['Que']
      ans = value['Ans']
      QID_context_q[QID] = ques + ' [SEP] ' + present_context
      QID_ans[QID] = ans


100%|██████████| 500/500 [00:00<00:00, 36081.28it/s]


In [ ]:
with open('/content/drive/MyDrive/MedQA/QID_q_context.pkl', 'wb') as file:
    pickle.dump(QID_context_q, file)

with open('/content/drive/MyDrive/MedQA/QID_ans.pkl', 'wb') as file:
    pickle.dump(QID_ans, file)